# Low-rank approximation with the SVD
Rank-k approximations of a small matrix and of a photo, the singular-value decay, and noise reduction.
The photo is scikit-learn's bundled sample image `china.jpg` (no download needed).

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.datasets import load_sample_image
np.set_printoptions(precision=3, suppress=True)

## 1. A matrix as a sum of rank-1 layers

In [ ]:
A = np.array([[3, 0], [4, 5]])
U, s, Vt = np.linalg.svd(A)
# layer i is sigma_i times the outer product u_i v_i^T
layers = [s[i] * np.outer(U[:, i], Vt[i]) for i in range(2)]
print(layers[0])          # the rank-1 approximation
print(layers[1])          # what it misses
print(layers[0] + layers[1])  # gives back A

## 2. Eckart-Young: the error of the best rank-1 approximation is sigma_2

In [ ]:
A1 = layers[0]
# ord=2 is the spectral norm: the largest stretch
print('SVD rank 1 error:', np.linalg.norm(A - A1, 2), ' sigma_2:', s[1])
# another rank-1 matrix: keep only the second row of A
B = np.array([[0, 0], [4, 5]])
print('other rank 1 error:', np.linalg.norm(A - B, 2))

## 3. A photo as a matrix

In [ ]:
rgb = load_sample_image('china.jpg')
# grayscale: weighted sum of red, green, blue, scaled to 0..1
img = rgb.astype(float) @ [0.299, 0.587, 0.114] / 255
m, n = img.shape
print(m, n, 'numbers:', m * n)
U, s, Vt = np.linalg.svd(img, full_matrices=False)
print('first singular values:', s[:5])

In [ ]:
def rank_k(k):
    # first k columns of U, scaled by sigma, times first k rows of V^T
    return (U[:, :k] * s[:k]) @ Vt[:k]

for k in [1, 5, 20, 50, 100]:
    stored = k * (m + n + 1)
    print(f'k={k:3d}  stores {stored:6d} numbers ({stored / (m * n):.1%})'
          f'  error sigma_k+1 / sigma_1 = {s[k] / s[0]:.1%}')

### Slide through the ranks

In [ ]:
ks = [1, 2, 5, 10, 20, 50, 100, 200, 427]
frames = [go.Frame(data=[go.Heatmap(z=rank_k(k), colorscale='gray', zmin=0, zmax=1, showscale=False)], name=str(k))
          for k in ks]
fig = go.Figure(frames[0].data, frames=frames)
steps = [dict(method='animate', label=str(k), args=[[str(k)], dict(mode='immediate', frame=dict(duration=0))])
         for k in ks]
fig.update_layout(sliders=[dict(steps=steps, currentvalue=dict(prefix='rank k = '))],
                  width=760, height=600, yaxis=dict(autorange='reversed', scaleanchor='x', visible=False),
                  xaxis=dict(visible=False))
fig.show()

## 4. How fast the singular values fall

In [ ]:
share = np.cumsum(s ** 2) / np.sum(s ** 2)
for k in [1, 5, 20, 50]:
    print(f'first {k:2d} singular values keep {share[k - 1]:.1%} of the sum of squares')
go.Figure(go.Scatter(y=s, mode='lines')).update_yaxes(type='log', title='sigma_i').update_xaxes(title='i').show()

## 5. Noise reduction

In [ ]:
clean = np.zeros((120, 160))
clean[50:70, 10:150] += 0.5      # horizontal bar
clean[10:110, 70:90] += 0.5      # vertical bar
clean[20:45, 20:55] += 1.0       # square
print('rank of the clean picture:', np.linalg.matrix_rank(clean))
noisy = clean + np.random.default_rng(0).normal(0, 0.3, clean.shape)
Un, sn, Vnt = np.linalg.svd(noisy, full_matrices=False)
print('singular values of the noisy picture:', sn[:6])
err = lambda X: np.linalg.norm(X - clean) / np.linalg.norm(clean)
print('noisy picture error:', round(err(noisy), 3))
for k in [1, 2, 3, 4, 10, 50]:
    print(k, round(err((Un[:, :k] * sn[:k]) @ Vnt[:k]), 3))